In [1]:
import rebound
import reboundx
import numpy as np
import matplotlib.pyplot as plt

In [6]:
rebound.__version__

'4.4.2'

In [3]:
date='2024-01-01 00:00'
sim = rebound.Simulation()
sim.add('Sun', date=date, hash='sun')
sim.add('Mercury', date=date)
sim.add('Venus', date=date)
sim.add('Earth', date=date, hash='earth')
sim.add('Mars', date=date)
sim.add('Jupiter', date=date)
sim.add('Saturn', date=date)
sim.add('Uranus', date=date)
sim.add('Neptune', date=date)
sim.move_to_com()
sim.convert_particle_units('AU', 'year', 'Msun')
sim.save_to_file('ss.bin')

Searching NASA Horizons for 'Sun'... 
Found: Sun (10) 
Searching NASA Horizons for 'Mercury'... 
Found: Mercury Barycenter (199) (chosen from query 'Mercury')
Searching NASA Horizons for 'Venus'... 
Found: Venus Barycenter (299) (chosen from query 'Venus')
Searching NASA Horizons for 'Earth'... 
Found: Earth-Moon Barycenter (3) (chosen from query 'Earth')
Searching NASA Horizons for 'Mars'... 
Found: Mars Barycenter (4) (chosen from query 'Mars')
Searching NASA Horizons for 'Jupiter'... 
Found: Jupiter Barycenter (5) (chosen from query 'Jupiter')
Searching NASA Horizons for 'Saturn'... 
Found: Saturn Barycenter (6) (chosen from query 'Saturn')
Searching NASA Horizons for 'Uranus'... 
Found: Uranus Barycenter (7) (chosen from query 'Uranus')
Searching NASA Horizons for 'Neptune'... 
Found: Neptune Barycenter (8) (chosen from query 'Neptune')


In [4]:
%%time
tmax = 1e5
Nsnaps = 1000

sim= rebound.Simulation('ss.bin')
sim.integrator = "WHCKL" 
sim.ri_whfast.safe_mode = False
sim.ri_whfast.corrector = 17
sim.ri_whfast.keep_unsynchronized=True
sim.dt = 4.062/365.25
sim.save_to_file('archive.bin', interval=int(tmax/Nsnaps), delete_file=True)

rebx = reboundx.Extras(sim)
gr = rebx.load_force('gr_potential')
rebx.add_force(gr)
gr.params['c'] = 63240 # speed of light in AU/yr

sim.integrate(tmax)

CPU times: user 16.5 s, sys: 32 ms, total: 16.5 s
Wall time: 16.5 s


In [5]:
sa = rebound.Simulationarchive("archive.bin")
sim = sa[0]
rebx = reboundx.Extras(sim)
gr = rebx.load_force('gr_potential')
rebx.add_force(gr)
gr.params['c'] = 63240 # speed of light in AU/yr
E0 = sim.energy() + rebx.gr_potential_potential(gr)
Eerr, times = np.zeros(len(sa)), np.zeros(len(sa))

for i, sim in enumerate(sa):
    rebx = reboundx.Extras(sim)
    gr = rebx.load_force('gr_potential')
    rebx.add_force(gr)
    gr.params['c'] = 63240 # speed of light in AU/yr

    sim.synchronize()
    E = sim.energy() + rebx.gr_potential_potential(gr)
    Eerr[i] = np.abs((E-E0)/E0)
    times[i] = sim.t

/home/miniconda3/envs/ahekster/lib/python3.12/site-packages/rebound/simulationarchive.py:150: RuntimeWarning: You have to reset function pointers after creating a reb_simulation struct with a binary file.
  warnings.warn(message, RuntimeWarning)


AttributeError: 'Simulation' object has no attribute 'synchronize'

In [ ]:
fig, ax = plt.subplots()
ax.plot(times[:-1], Eerr[:-1], '.')
ax.set_xscale('log')
ax.set_yscale('log')

# Reproducibility

In [ ]:
t = sa[2].t
sim = sa[2]

# Have to readd forces and parameters
rebx = reboundx.Extras(sim)
gr = rebx.load_force('gr_potential')
rebx.add_force(gr)
gr.params['c'] = 63240 # speed of light in AU/yr

sim.synchronize()
sim.particles[1].x

In [ ]:
sim = sa[1]

# Have to readd forces and parameters
rebx = reboundx.Extras(sim)
gr = rebx.load_force('gr_potential')
rebx.add_force(gr)
gr.params['c'] = 63240 # speed of light in AU/yr

sim.integrate(t, exact_finish_time=0)
sim.synchronize()
sim.particles[1].x